## Data Preparation

This notebook prepares the Amazon review dataset for training sentiment analysis models.

Steps performed:

1. Load dataset
2. Map star ratings to sentiment labels
3. Balance the dataset
4. Encode labels for machine learning
5. Split into training and testing sets
6. Save processed datasets

In [2]:
#Import required libraries
import pandas as pd
from sklearn.model_selection import train_test_split

#Load dataset
path = r"../data/Reviews.csv"

df = pd.read_csv(path)

#Select only the columns needed for sentiment analysis
df = df[["Score", "Text"]] #Score = star rating and Text = review content

def map_sentiment(score):
    """ Convert Amazon star ratings into sentiment categories
    Parameters
    score : int
        Start rating from the review (1-5)
    Returns
    str
        Sentiment label: negative, neutral, positive.
    """
    
    if score <= 2:
        return "negative"
    elif score == 3:
        return "neutral"
    else:
        return "positive"

df["sentiment"] = df["Score"].apply(map_sentiment)

df.head()

,Score,Text,sentiment
0,5,I have bought several of the Vitality canned d...,positive
1,1,Product arrived labeled as Jumbo Salted Peanut...,negative
2,4,This is a confection that has been around a fe...,positive
3,2,If you are looking for the secret ingredient i...,negative
4,5,Great taffy at a great price. There was a wid...,positive


In [3]:
#Inspect distrtibutin of sentiment labels
sentiment_counts = df["sentiment"].value_counts()
print(sentiment_counts)

sentiment
positive    443777
negative     82037
neutral      42640
Name: count, dtype: int64


In [4]:
#Balance Dataset
#Each class is sampled to mathch the size of smallest class

positive_df = df[df["sentiment"] == "positive"]
negative_df = df[df["sentiment"] == "negative"]
neutral_df  = df[df["sentiment"] == "neutral"]

#Calculate size of smallest class
min_size = min(len(positive_df), len(negative_df), len(neutral_df)) #

#Set to size to smallest class
positive_sample = positive_df.sample(n=min_size, random_state=42)
negative_sample = negative_df.sample(n=min_size, random_state=42)
neutral_sample  = neutral_df.sample(n=min_size, random_state=42)

#Combine reviews into a balanced dataset
balanced_df = pd.concat([positive_sample, negative_sample, neutral_sample])

#Shuffle Dataset to mix classes
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(balanced_df.head())
print(" ")
print("Sentiment count after balance : ")
balanced_df["sentiment"].value_counts()

   Score                                               Text sentiment
0      2  I was disappointed with the quality of the cak...  negative
1      1  I was hoping for better! The tea has a burnt f...  negative
2      5  We have a toothless older small dog and this i...  positive
3      1  Chinese made products are beyond questionable ...  negative
4      1  Save your money.  Go to a local green house Th...  negative
 
Sentiment count after balance : 


sentiment
negative    42640
positive    42640
neutral     42640
Name: count, dtype: int64

In [5]:
#Convert sentiment labels into numeric format for model training
label_mapping = {
        "negative": 0,
    "neutral": 1,
    "positive": 2
}

balanced_df["label"] = balanced_df["sentiment"].map(label_mapping)

balanced_df.head()

,Score,Text,sentiment,label
0,2,I was disappointed with the quality of the cak...,negative,0
1,1,I was hoping for better! The tea has a burnt f...,negative,0
2,5,We have a toothless older small dog and this i...,positive,2
3,1,Chinese made products are beyond questionable ...,negative,0
4,1,Save your money. Go to a local green house Th...,negative,0


In [6]:
#Split dataset into training and testing data

train_df, test_df = train_test_split(
    balanced_df,
    test_size=0.2,
    stratify=balanced_df["label"],
    random_state=42
)

#Save datasets for model training
train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)



Train shape: (102336, 4)
Test shape: (25584, 4)
